In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
# ── Cell 2: Read Silver weather ───────────────────────────────
log_header("ml_features", layer="GOLD")
print("\n[1/3] Reading Silver weather...")

df_silver = spark.table(f"{silver_catalog}.weather_readings")
print(f"  Silver rows: {df_silver.count():,}")

In [0]:
# ── Cell 3: Build ML feature table ───────────────────────────
print("\n[2/3] Engineering features...")

window_city = Window.partitionBy("city_name").orderBy("reading_timestamp")

gold_ml_features = (
    df_silver
    # ── Lag features (historical context) ────────────────────
    .withColumn("temp_lag_1h",
        F.lag("temperature_c", 1).over(window_city))
    .withColumn("temp_lag_3h",
        F.lag("temperature_c", 3).over(window_city))
    .withColumn("temp_lag_6h",
        F.lag("temperature_c", 6).over(window_city))
    .withColumn("temp_lag_24h",
        F.lag("temperature_c", 24).over(window_city))
    .withColumn("humidity_lag_1h",
        F.lag("humidity_pct", 1).over(window_city))
    .withColumn("pressure_lag_1h",
        F.lag("pressure_hpa", 1).over(window_city))
    .withColumn("wind_lag_1h",
        F.lag("wind_speed_ms", 1).over(window_city))

    # ── Trend features ────────────────────────────────────────
    .withColumn("pressure_trend_3h",
        F.col("pressure_hpa") -
        F.lag("pressure_hpa", 3).over(window_city))
    .withColumn("temp_trend_3h",
        F.col("temperature_c") -
        F.lag("temperature_c", 3).over(window_city))

    # ── Target variable: temp in 24 hours ────────────────────
    .withColumn("temp_target_24h",
        F.lead("temperature_c", 24).over(window_city))

    # ── Keep rows where we have enough history ────────────────
    .filter(F.col("temp_lag_1h").isNotNull())

    # ── Select ML feature columns only ───────────────────────
    .select(
        "city_name", "reading_timestamp", "reading_date",
        "latitude", "longitude", "reading_hour",
        "temperature_c", "humidity_pct", "pressure_hpa",
        "wind_speed_ms", "cloud_cover_pct",
        "is_daytime", "wind_beaufort",
        "temp_lag_1h", "temp_lag_3h",
        "temp_lag_6h", "temp_lag_24h",
        "humidity_lag_1h", "pressure_lag_1h", "wind_lag_1h",
        "pressure_trend_3h", "temp_trend_3h",
        "temp_target_24h"
    )
    .withColumn("ingestion_date", F.current_date())
    .withColumn("ingestion_ts",   F.current_timestamp())
)

row_count = gold_ml_features.count()
print(f"  ML feature rows: {row_count:,}")
print(f"  Rows with target: {gold_ml_features.filter(F.col('temp_target_24h').isNotNull()).count():,}")

In [0]:
# ── Cell 4: Write to ADLS Gen2 + register in Unity Catalog ───
print("\n[3/3] Writing to ADLS Gen2...")
row_count = write_gold_table(gold_ml_features, "ml_features", partition_col="reading_date")

In [0]:
# ── Cell 5: Preview ───────────────────────────────────────────
spark.sql(f"""
    SELECT city_name, reading_timestamp,
           temperature_c, temp_lag_1h, temp_lag_3h,
           pressure_trend_3h, temp_target_24h
    FROM {gold_catalog}.ml_features
    ORDER BY city_name, reading_timestamp
    LIMIT 10
""").show(truncate=False)

log_footer("ml_features", row_count, "PASS", layer="GOLD")
dbutils.notebook.exit(f"ml_features|{row_count}|PASS")